# Scan-like threshold exploration

This notebook visualizes threshold-free, 60 s source-window observations for the M4 Human Gate. It does not classify traffic or select thresholds.

## 1. Load data

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
from IPython.display import display

from mawi_global_analysis.io import load_run

root = Path(os.environ.get('MAWI_ANALYSIS_ROOT', '.')).resolve()
dataset_id = os.environ.get('MAWI_DATASET_ID', 'fixture')
run_name = os.environ.get('MAWI_RUN_NAME', 'baseline')
run = load_run(dataset_id, run_name, root=root)
if run.scan_windows is None:
    raise RuntimeError('threshold exploration requires source_scan_windows.csv')

windows = run.scan_windows.copy()
display(windows.head())

FileNotFoundError: run manifest not found: /Users/shunta/mawi-global-analysis/notebooks/results/fixture/baseline/run_manifest.json

## 2. Broad Scan-like scatter

Each point is one source-window. Use this view to inspect bulk activity, the tail, isolated source-windows, and possible distribution breaks.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    windows['syn_initiated_flow_count'],
    windows['unique_targets'],
    alpha=0.35,
)
ax.set(
    xscale='log',
    yscale='log',
    xlabel='SYN-initiated flows per 60 s window',
    ylabel='Unique targets per 60 s window',
    title='Broad scan-like source-window activity',
)
plt.show()

## 3. Strict / High-confidence scatter

Each point is one source-window with positive values on both axes. `high_confidence_probe_pattern_count` is based on the positive observed TCP patterns `syn_to_rst` (not `sin_to_rst`) and `syn_synack_rst`, following the repository implementation.

In [ ]:
high_confidence = windows.loc[
    (windows['high_confidence_probe_pattern_count'] > 0)
    & (windows['unique_high_confidence_targets'] > 0)
].copy()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    high_confidence['high_confidence_probe_pattern_count'],
    high_confidence['unique_high_confidence_targets'],
    alpha=0.35,
    color='tab:orange',
)
ax.set(
    xscale='log',
    yscale='log',
    xlabel='High-confidence probe patterns per 60 s window',
    ylabel='Unique high-confidence targets per 60 s window',
    title='Strict / high-confidence source-window activity',
)
plt.show()

## M4 Human Gate

These two scatter plots are for visual review of threshold candidates at the Human Gate. Thresholds are not selected automatically.